# 18b — Initial Target Scoring V2: Calibrated Watchlists

This notebook is a revised scoring pass. It keeps the original V1 scoring outputs intact and writes all V2 files to a separate folder.

Main changes from V1:

1. **Demographic relevance is rescaled to a true 0–100 range.** In V1, the demographic component was capped too low, which let electoral opportunity dominate the combined score.
2. **Clean and caveated watchlists are split.** County-election-derived rows, boundary caveats and invalid vote issues are kept visible instead of mixed into the clean list.
3. **A demographic build list is created.** This catches places that look socially/demographically aligned but are electorally harder.
4. **A breakthrough/complacency list is created.** This recognises the political reality that apparently “safe” major-party wards can still become breakthrough opportunities when the demographic terrain is plausible, turnout is low, and local dissatisfaction can be mobilised.

The output is still a **watchlist model**, not a final targeting decision model. Candidate availability, local party strength, members, branch capacity and live local issues still need to be added later.


## 18b.1 Project paths and switches

Default scope remains **North West**, but the notebook can also generate all-available and region outputs. Keep `GENERATE_REGION_OUTPUTS = False` until you need regional batch exports.


In [8]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import json

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name.lower() == "notebooks" else NOTEBOOK_DIR

DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"

V1_MODEL_DIR = PROCESSED_DIR / "target_model_v1"
V1_INPUT_DIR = V1_MODEL_DIR / "inputs"
V1_OUTPUT_DIR = V1_MODEL_DIR / "outputs"

V2_MODEL_DIR = PROCESSED_DIR / "target_model_v2"
V2_INPUT_DIR = V2_MODEL_DIR / "inputs"
V2_OUTPUT_DIR = V2_MODEL_DIR / "outputs"
V2_REVIEW_DIR = V2_MODEL_DIR / "review"
V2_WATCHLIST_DIR = V2_MODEL_DIR / "watchlists"

for d in [V2_INPUT_DIR, V2_OUTPUT_DIR, V2_REVIEW_DIR, V2_WATCHLIST_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Scope controls.
DEFAULT_SCOPE_COLUMN = "scope_north_west"
DEFAULT_SCOPE_NAME = "north_west"

GENERATE_ALL_AVAILABLE_OUTPUTS = True
GENERATE_REGION_OUTPUTS = True

# Combined score weights.
SCORE_WEIGHTS = {
    "demographic_relevance_score": 0.35,
    "electoral_opportunity_score": 0.30,
    "political_openness_score": 0.25,
    "data_confidence_score": 0.10,
}

print("Project:", PROJECT_DIR)
print("V1 input folder:", V1_INPUT_DIR)
print("V2 output folder:", V2_OUTPUT_DIR)


Project: c:\Users\keena\Documents\Electoral_Tribes
V1 input folder: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_model_v1\inputs
V2 output folder: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_model_v2\outputs


## 18b.2 Load model input

This uses the target model input created by Notebook 17. V2 does not alter that input; it only recalculates score components and outputs new watchlists.


In [9]:
INPUT_PATH = V1_INPUT_DIR / "target_model_input_ward25_v1.csv"
NW_INPUT_PATH = V1_INPUT_DIR / "target_model_input_north_west_ward25_v1.csv"

if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Missing model input: {INPUT_PATH}. Run Notebook 17 first.")

df = pd.read_csv(INPUT_PATH, low_memory=False)

# Keep a copy of the exact input used for V2.
df.to_csv(V2_INPUT_DIR / "target_model_input_ward25_used_for_v2.csv", index=False)

print("Rows:", len(df))
print("Columns:", len(df.columns))
display(df.head())


Rows: 7572
Columns: 119


,LAD25CD,LAD25NM,WD25CD,WD25NM,RGN25CD,RGN25NM,analysis_region,country_inferred,scope_north_west,scope_england,scope_wales,scope_all_available,population,oa_count,dominant_cluster,dominant_cluster_name,second_cluster,second_cluster_name,dominant_cluster_share,second_cluster_share,cluster_fragmentation_index,is_mixed_ward,is_clear_dominant_ward,is_highly_fragmented,cluster_0_share,cluster_1_share,cluster_2_share,cluster_3_share,cluster_4_share,cluster_5_share,cluster_6_share,student_transient_youth_share,rooted_older_homeowners_share,stable_suburban_professionals_share,cosmopolitan_young_professional_core_share,settled_working_families_skilled_trades_suburbs_share,settled_diverse_urban_communities_share,post_industrial_estates_deprived_working_communities_share,age_0_14_pct,age_15_24_pct,age_25_34_pct,age_35_49_pct,age_50_64_pct,age_65_plus_pct,uk_born_pct,non_uk_born_pct,resident_10_plus_years_pct,resident_less_5_years_pct,white_british_pct,white_other_pct,non_white_pct,owned_pct,owns_outright_pct,owns_mortgage_pct,social_rented_pct,private_rented_pct,house_type_pct,flat_type_pct,managerial_professional_pct,skilled_traditional_pct,routine_service_elementary_pct,employed_pct,unemployed_pct,full_time_student_pct,retired_pct,long_term_sick_disabled_pct,no_qualifications_pct,level_1_2_pct,apprenticeship_pct,level_4_plus_pct,one_person_household_pct,married_couple_family_pct,lone_parent_family_pct,latest_election_WD25NM,latest_election_LAD25CD,latest_election_LAD25NM,latest_election_source_year,latest_election_allocated_electorate,latest_election_allocated_valid_votes,latest_election_allocated_ballots,latest_election_allocated_invalid_votes,latest_election_allocated_top_party_votes,latest_election_allocated_runner_up_party_votes,latest_election_allocated_con_votes,latest_election_allocated_lab_votes,latest_election_allocated_ld_votes,latest_election_allocated_green_votes,latest_election_allocated_reform_ukip_brexit_votes,latest_election_allocated_independent_votes,latest_election_allocated_sdp_votes,latest_election_allocated_other_votes,latest_election_contributing_oa_rows,latest_election_contributing_result_areas,latest_election_contributing_source_years,latest_election_con_share,latest_election_lab_share,latest_election_ld_share,latest_election_green_share,latest_election_reform_ukip_brexit_share,latest_election_independent_share,latest_election_sdp_share,latest_election_other_share,latest_election_top_party_bucket,latest_election_runner_up_party_bucket,latest_election_top_party_votes_allocated,latest_election_runner_up_party_votes_allocated,latest_election_margin_votes_allocated,latest_election_margin_pct_allocated,latest_election_party_fragmentation_index,latest_election_effective_number_of_parties,latest_election_aggregation_label,latest_election_latest_layer_note,has_latest_election_layer,has_valid_vote_data,has_margin_data,boundary_caveat,county_election_caveat,target_model_ready,data_confidence_note
0,E06000001,Hartlepool,E05013038,Burn Valley,E12000001,North East,North East,England,False,True,False,True,7633,26,6.0,Post-Industrial Estates / Deprived Working Com...,2.0,Stable Suburban Professionals,0.458404,0.191406,0.703925,False,False,True,0.043495,0.167169,0.191406,0.0,0.139526,0.0,0.458404,0.043495,0.167169,0.191406,0.0,0.139526,0.0,0.458404,0.174112,0.137037,0.106511,0.179353,0.206996,0.194157,0.948513,0.051487,0.029208,0.014931,0.925465,0.018863,0.052135,0.554228,0.304751,0.249477,0.217807,0.224081,0.843032,0.155774,0.371084,0.207229,0.333219,0.467880,0.046208,0.092256,0.234262,0.084688,0.225000,0.240806,0.062903,0.267419,0.389420,0.251046,0.124626,Burn Valley,E06000001,Hartlepool,2024.0,5818.0,1686.0,0.0,0.0,919.0,339.0,339.0,919.0,0.0,0.0,270.0,158.0,0.0,0.0,26.0,1.0,2024.0,0.201068,0.545077,0.0,0.0,0.160142,0.093713,0.0,0.000000,lab,con,919.0,339.0,580.0,0.344009,0.628035,2.688426,ward25_by_source_year,latest_available_source_year_after_oa21_crosswalk,True,True,True,NaN,NaN,True,No major caveat.
1,E06000001,Hartlep

## 18b.3 Helper functions

Scores are scaled 0–100, where higher means stronger fit to that component.


In [10]:
def to_num(s):
    return pd.to_numeric(s, errors="coerce")


def clip100(s):
    return to_num(s).replace([np.inf, -np.inf], np.nan).fillna(0).clip(0, 100)


def safe_col(frame, col, default=0):
    if col in frame.columns:
        return to_num(frame[col]).replace([np.inf, -np.inf], np.nan).fillna(default)
    return pd.Series(default, index=frame.index, dtype=float)


def safe_bool_col(frame, col, default=False):
    if col not in frame.columns:
        return pd.Series(default, index=frame.index, dtype=bool)
    s = frame[col]
    if s.dtype == bool:
        return s.fillna(default)
    return s.astype(str).str.strip().str.lower().isin(["true", "1", "yes", "y"])


def has_text_caveat(series):
    return series.notna() & series.astype(str).str.strip().ne("") & series.astype(str).str.lower().ne("nan")


def pct_to_score_share(s):
    """Convert a vote share/proportion in 0–1 into a 0–100 score."""
    return (to_num(s).fillna(0).clip(0, 1) * 100).clip(0, 100)


def save_csv(frame, folder, filename):
    path = folder / filename
    frame.to_csv(path, index=False)
    print("Saved:", path)
    return path


def scope_filename(scope_name, stem, version="v2"):
    safe = re.sub(r"[^a-zA-Z0-9]+", "_", scope_name.lower()).strip("_")
    return f"{stem}_{safe}_{version}.csv"


def filter_scope(frame, scope_name, scope_column=None):
    if scope_column and scope_column in frame.columns:
        return frame[frame[scope_column].fillna(False).astype(bool)].copy()
    if scope_name == "all_available":
        if "scope_all_available" in frame.columns:
            return frame[frame["scope_all_available"].fillna(False).astype(bool)].copy()
        return frame.copy()
    if "analysis_region" in frame.columns:
        return frame[frame["analysis_region"].astype(str).str.lower().eq(scope_name.replace("_", " ").lower())].copy()
    return frame.copy()


## 18b.4 Build calibrated score components

The revised demographic score is now scaled so that a ward dominated by the strongest-fit demographic cluster can score close to 100. This prevents the demographic component being mechanically weaker than the electoral components.

A separate **breakthrough/complacency score** is also calculated. This is not treated as conventional electoral opportunity. It exists because a “safe” major-party ward can still become politically live when the demographic terrain is plausible and turnout/apathy suggests latent dissatisfaction.


In [11]:
scored = df.copy()

# -------------------------
# Core shares and election values
# -------------------------
post_industrial = safe_col(scored, "post_industrial_estates_deprived_working_communities_share")
settled_working = safe_col(scored, "settled_working_families_skilled_trades_suburbs_share")
rooted_older = safe_col(scored, "rooted_older_homeowners_share")
stable_suburban = safe_col(scored, "stable_suburban_professionals_share")
settled_diverse = safe_col(scored, "settled_diverse_urban_communities_share")
student_transient = safe_col(scored, "student_transient_youth_share")
cosmopolitan_core = safe_col(scored, "cosmopolitan_young_professional_core_share")

valid_votes = safe_col(scored, "latest_election_allocated_valid_votes")
electorate = safe_col(scored, "latest_election_allocated_electorate")
top_party_votes = safe_col(scored, "latest_election_top_party_votes_allocated")
runner_up_votes = safe_col(scored, "latest_election_runner_up_party_votes_allocated")
margin_pct = safe_col(scored, "latest_election_margin_pct_allocated")
fragmentation = safe_col(scored, "latest_election_party_fragmentation_index")
effective_parties = safe_col(scored, "latest_election_effective_number_of_parties")

con_share = safe_col(scored, "latest_election_con_share")
lab_share = safe_col(scored, "latest_election_lab_share")
ld_share = safe_col(scored, "latest_election_ld_share")
green_share = safe_col(scored, "latest_election_green_share")
reform_share = safe_col(scored, "latest_election_reform_ukip_brexit_share")
independent_share = safe_col(scored, "latest_election_independent_share")
sdp_share = safe_col(scored, "latest_election_sdp_share")
other_share = safe_col(scored, "latest_election_other_share")

# -------------------------
# Demographic relevance score — V2 calibration
# -------------------------
# Positive cluster weights are deliberately not equal. The score is scaled by
# the maximum theoretical positive weight (40), so a strongly post-industrial
# ward can approach 100 rather than being capped at ~40–45 as in V1.
demo_raw = (
    40 * post_industrial
    + 30 * settled_working
    + 20 * rooted_older
    + 10 * stable_suburban
    + 5 * settled_diverse
    - 10 * student_transient
    - 10 * cosmopolitan_core
)

scored["demographic_relevance_raw_v2"] = demo_raw
scored["demographic_relevance_score"] = (demo_raw.clip(lower=0) / 40 * 100).clip(0, 100)

# -------------------------
# Electoral opportunity score
# -------------------------
# Lower margins, weaker top-party dominance and lower vote thresholds score higher.
top_party_share = np.where(valid_votes > 0, top_party_votes / valid_votes, np.nan)
scored["latest_election_top_party_share"] = pd.Series(top_party_share, index=scored.index).fillna(0).clip(0, 1)

# Margin of 0% = 100, margin of 40%+ = 0.
scored["margin_competitiveness_score"] = (100 * (1 - margin_pct.abs() / 0.40)).clip(0, 100)

# Top party dominance inverse: lower top-party share = more competitive.
scored["top_party_dominance_inverse_score"] = (100 * (1 - scored["latest_election_top_party_share"])).clip(0, 100)

# Lower top-party vote threshold is operationally easier to overcome.
# 2,000 top-party votes or more is scored as 0 for this component.
scored["vote_threshold_score"] = (100 * (1 - top_party_votes / 2000)).clip(0, 100)

# Fragmentation: 0.75 is roughly the upper practical range for a very fragmented contest.
scored["electoral_fragmentation_score"] = (fragmentation / 0.75 * 100).clip(0, 100)

scored["electoral_opportunity_score"] = (
    0.40 * scored["margin_competitiveness_score"]
    + 0.25 * scored["top_party_dominance_inverse_score"]
    + 0.25 * scored["vote_threshold_score"]
    + 0.10 * scored["electoral_fragmentation_score"]
).clip(0, 100)

# -------------------------
# Political openness score
# -------------------------
non_main_party_share = (reform_share + independent_share + sdp_share + other_share + green_share).clip(0, 1)
challenger_share = (ld_share + green_share + reform_share + independent_share + sdp_share + other_share).clip(0, 1)
lab_con_share = (lab_share + con_share).clip(0, 1)

scored["non_main_party_score"] = (non_main_party_share * 100).clip(0, 100)
scored["challenger_party_score"] = (challenger_share * 100).clip(0, 100)
scored["lab_con_inverse_score"] = (100 * (1 - lab_con_share)).clip(0, 100)
scored["effective_parties_score"] = ((effective_parties - 1) / 3 * 100).clip(0, 100)

# -------------------------
# Breakthrough / complacency score
# -------------------------
# This captures the possibility that a safe major-party ward can still become
# live where turnout/apathy is low and the demographic terrain is plausible.
turnout_proxy = np.where(electorate > 0, valid_votes / electorate, np.nan)
scored["latest_election_turnout_proxy"] = pd.Series(turnout_proxy, index=scored.index).replace([np.inf, -np.inf], np.nan)
scored["low_turnout_apathy_score"] = (100 * (1 - scored["latest_election_turnout_proxy"] / 0.45)).clip(0, 100).fillna(0)

latest_top_party = scored.get("latest_election_top_party_bucket", pd.Series("", index=scored.index)).astype(str).str.lower().str.strip()
major_party_top = latest_top_party.isin(["lab", "con", "labour", "conservative"])

scored["major_party_safe_seat_score"] = np.where(
    major_party_top,
    scored["latest_election_top_party_share"] * 100,
    0,
)

scored["breakthrough_complacency_score"] = np.where(
    major_party_top,
    (
        0.40 * scored["demographic_relevance_score"]
        + 0.35 * scored["major_party_safe_seat_score"]
        + 0.25 * scored["low_turnout_apathy_score"]
    ),
    0,
).clip(0, 100)

scored["political_openness_score"] = (
    0.30 * scored["non_main_party_score"]
    + 0.25 * scored["lab_con_inverse_score"]
    + 0.20 * scored["effective_parties_score"]
    + 0.15 * scored["challenger_party_score"]
    + 0.10 * scored["breakthrough_complacency_score"]
).clip(0, 100)

# -------------------------
# Data confidence score
# -------------------------
has_latest = safe_bool_col(scored, "has_latest_election_layer")
has_valid_vote = safe_bool_col(scored, "has_valid_vote_data")
has_margin = safe_bool_col(scored, "has_margin_data")
boundary_caveat = has_text_caveat(scored.get("boundary_caveat", pd.Series(np.nan, index=scored.index)))
county_caveat = has_text_caveat(scored.get("county_election_caveat", pd.Series(np.nan, index=scored.index)))

latest_year = to_num(scored.get("latest_election_source_year", pd.Series(np.nan, index=scored.index)))
recency_penalty = np.select(
    [latest_year <= 2022, latest_year == 2023, latest_year >= 2024],
    [10, 5, 0],
    default=20,
)

scored["has_major_caveat"] = (
    (~has_latest)
    | (~has_valid_vote)
    | (~has_margin)
    | boundary_caveat
    | county_caveat
)

scored["data_confidence_score"] = (
    100
    - np.where(~has_latest, 30, 0)
    - np.where(~has_valid_vote, 30, 0)
    - np.where(~has_margin, 20, 0)
    - np.where(boundary_caveat, 15, 0)
    - np.where(county_caveat, 20, 0)
    - recency_penalty
)
scored["data_confidence_score"] = scored["data_confidence_score"].clip(0, 100)

# -------------------------
# Combined score
# -------------------------
scored["initial_watchlist_score"] = sum(
    SCORE_WEIGHTS[col] * scored[col]
    for col in SCORE_WEIGHTS
).clip(0, 100)

# Percentiles across all available rows.
scored["initial_watchlist_percentile"] = scored["initial_watchlist_score"].rank(pct=True) * 100

scored["review_band"] = pd.cut(
    scored["initial_watchlist_percentile"],
    bins=[-0.01, 70, 85, 95, 100],
    labels=["Review D", "Review C", "Review B", "Review A"],
).astype(str)

scored["review_band_clean"] = np.where(
    scored["has_major_caveat"],
    scored["review_band"] + " Caveated",
    scored["review_band"] + " Clean",
)

component_cols = [
    "LAD25CD", "LAD25NM", "WD25CD", "WD25NM", "analysis_region",
    "dominant_cluster_name", "second_cluster_name",
    "initial_watchlist_score", "initial_watchlist_percentile", "review_band", "review_band_clean",
    "demographic_relevance_score", "electoral_opportunity_score", "political_openness_score", "data_confidence_score",
    "margin_competitiveness_score", "top_party_dominance_inverse_score", "vote_threshold_score", "electoral_fragmentation_score",
    "non_main_party_score", "challenger_party_score", "effective_parties_score", "lab_con_inverse_score",
    "breakthrough_complacency_score", "major_party_safe_seat_score", "low_turnout_apathy_score",
    "latest_election_source_year", "latest_election_top_party_bucket", "latest_election_runner_up_party_bucket",
    "latest_election_margin_pct_allocated", "latest_election_top_party_share", "latest_election_turnout_proxy",
    "data_confidence_note", "boundary_caveat", "county_election_caveat", "has_major_caveat",
]
component_cols = [c for c in component_cols if c in scored.columns]

score_components = scored[component_cols].copy()

print("Score component summary:")
display(score_components[[
    "initial_watchlist_score", "demographic_relevance_score", "electoral_opportunity_score", "political_openness_score", "data_confidence_score",
    "breakthrough_complacency_score"
]].describe())


Score component summary:


,initial_watchlist_score,demographic_relevance_score,electoral_opportunity_score,political_openness_score,data_confidence_score,breakthrough_complacency_score
count,7572.000000,7572.000000,7572.000000,7572.000000,7572.000000,7572.000000
mean,54.034768,49.535842,55.543191,42.358935,94.445325,21.076363
std,12.385038,23.016992,20.051388,14.781447,11.235113,22.969360
min,15.536224,0.000000,0.000000,6.291515,0.000000,0.000000
25%,46.814758,36.600932,39.683675,30.774987,90.000000,0.000000
50%,55.699219,52.592245,58.481317,43.681077,95.000000,15.995378
75%,63.200384,65.667633,71.291885,53.383667,100.000000,40.561337
max,81.990184,100.000000,100.000000,79.384992,100.000000,83.157295


## 18b.5 Save all-available score components

The all-available file lets later notebooks compare nationally or filter by region without rerunning the model.


In [12]:
save_csv(score_components, V2_OUTPUT_DIR, "target_score_components_ward25_all_available_v2.csv")
save_csv(scored, V2_OUTPUT_DIR, "initial_watchlist_scores_ward25_all_available_v2.csv")

# Model parameter summary for audit.
parameter_rows = []
for k, v in SCORE_WEIGHTS.items():
    parameter_rows.append({"parameter_group": "combined_score_weight", "parameter": k, "value": v})

extra_params = {
    "demographic_scaling": "demo_raw clipped at 0, divided by 40, multiplied by 100",
    "margin_zero_score_threshold": "40% margin or greater scores 0 on margin competitiveness",
    "vote_threshold_zero_score": "2,000 top-party votes or greater scores 0 on vote threshold",
    "fragmentation_scaling": "party fragmentation / 0.75 * 100, clipped",
    "breakthrough_logic": "major-party top ward only; combines demographic relevance, major-party dominance, low turnout/apathy",
}
for k, v in extra_params.items():
    parameter_rows.append({"parameter_group": "rule", "parameter": k, "value": v})

parameters = pd.DataFrame(parameter_rows)
save_csv(parameters, V2_REVIEW_DIR, "initial_target_scoring_parameters_v2.csv")


Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_model_v2\outputs\target_score_components_ward25_all_available_v2.csv
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_model_v2\outputs\initial_watchlist_scores_ward25_all_available_v2.csv
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_model_v2\review\initial_target_scoring_parameters_v2.csv


WindowsPath('c:/Users/keena/Documents/Electoral_Tribes/data/processed/target_model_v2/review/initial_target_scoring_parameters_v2.csv')

## 18b.6 Scope outputs and watchlists

This section creates practical review files. Default scope is North West, but all-available outputs are also produced.


In [13]:
def make_council_summary(scope_df):
    if len(scope_df) == 0:
        return pd.DataFrame()

    summary = (
        scope_df.groupby(["LAD25CD", "LAD25NM", "analysis_region"], dropna=False, as_index=False)
        .agg(
            ward_count=("WD25CD", "nunique"),
            mean_score=("initial_watchlist_score", "mean"),
            max_score=("initial_watchlist_score", "max"),
            mean_demographic=("demographic_relevance_score", "mean"),
            mean_electoral=("electoral_opportunity_score", "mean"),
            mean_openness=("political_openness_score", "mean"),
            mean_data_confidence=("data_confidence_score", "mean"),
            review_a_count=("review_band", lambda x: (x == "Review A").sum()),
            review_b_count=("review_band", lambda x: (x == "Review B").sum()),
            clean_review_ab_count=("review_band_clean", lambda x: x.isin(["Review A Clean", "Review B Clean"]).sum()),
            caveated_review_ab_count=("review_band_clean", lambda x: x.isin(["Review A Caveated", "Review B Caveated"]).sum()),
            high_demo_count=("demographic_relevance_score", lambda x: (x >= 70).sum()),
            high_breakthrough_count=("breakthrough_complacency_score", lambda x: (x >= 65).sum()),
            major_caveat_count=("has_major_caveat", "sum"),
        )
    )

    # Simple triage category.
    conditions = [
        summary["clean_review_ab_count"] >= 3,
        (summary["clean_review_ab_count"] >= 1) | (summary["review_a_count"] >= 1),
        summary["high_demo_count"] >= 5,
        summary["major_caveat_count"] / summary["ward_count"] >= 0.4,
    ]
    choices = [
        "Priority exploration council",
        "Ward-level watchlist present",
        "Long-term demographic build",
        "Data caveat council",
    ]
    summary["council_triage"] = np.select(conditions, choices, default="Lower current priority")

    return summary.sort_values(["clean_review_ab_count", "mean_score"], ascending=[False, False])


def make_watchlists(scope_df, scope_name):
    scope_df = scope_df.copy()
    if len(scope_df) == 0:
        print("No rows for scope", scope_name)
        return

    # Full ranked score list.
    ranked = scope_df.sort_values("initial_watchlist_score", ascending=False)
    save_csv(ranked.head(100), V2_WATCHLIST_DIR, scope_filename(scope_name, "top_100_initial_watchlist", "v2"))

    # Component top lists.
    save_csv(scope_df.sort_values("demographic_relevance_score", ascending=False).head(50), V2_WATCHLIST_DIR, scope_filename(scope_name, "top_50_demographic_relevance", "v2"))
    save_csv(scope_df.sort_values("electoral_opportunity_score", ascending=False).head(50), V2_WATCHLIST_DIR, scope_filename(scope_name, "top_50_electoral_opportunity", "v2"))
    save_csv(scope_df.sort_values("political_openness_score", ascending=False).head(50), V2_WATCHLIST_DIR, scope_filename(scope_name, "top_50_political_openness", "v2"))

    # Clean/caveated watchlists.
    clean = scope_df[
        scope_df["review_band_clean"].isin(["Review A Clean", "Review B Clean"])
    ].sort_values("initial_watchlist_score", ascending=False)
    save_csv(clean, V2_WATCHLIST_DIR, scope_filename(scope_name, "clean_watchlist_review_ab", "v2"))

    caveated = scope_df[
        scope_df["review_band_clean"].isin(["Review A Caveated", "Review B Caveated"])
    ].sort_values("initial_watchlist_score", ascending=False)
    save_csv(caveated, V2_WATCHLIST_DIR, scope_filename(scope_name, "caveated_watchlist_review_ab", "v2"))

    # Demographic build: socially plausible but not necessarily electorally easy.
    demographic_build = scope_df[
        (scope_df["demographic_relevance_score"] >= 70)
        & (~scope_df["review_band_clean"].isin(["Review A Clean", "Review B Clean"]))
    ].sort_values(["demographic_relevance_score", "breakthrough_complacency_score"], ascending=False)
    save_csv(demographic_build.head(150), V2_WATCHLIST_DIR, scope_filename(scope_name, "demographic_build_watchlist", "v2"))

    # Breakthrough/complacency list: major-party stronghold, plausible terrain, possible apathy.
    breakthrough = scope_df[
        (scope_df["breakthrough_complacency_score"] >= 60)
        & (scope_df["demographic_relevance_score"] >= 45)
    ].sort_values("breakthrough_complacency_score", ascending=False)
    save_csv(breakthrough.head(150), V2_WATCHLIST_DIR, scope_filename(scope_name, "breakthrough_complacency_watchlist", "v2"))

    # Council summary.
    council_summary = make_council_summary(scope_df)
    save_csv(council_summary, V2_REVIEW_DIR, scope_filename(scope_name, "council_review_summary", "v2"))

    # Review band summaries.
    if "dominant_cluster_name" in scope_df.columns:
        band_cluster = (
            scope_df.groupby(["review_band_clean", "dominant_cluster_name"], dropna=False, as_index=False)
            .agg(ward_count=("WD25CD", "nunique"), mean_score=("initial_watchlist_score", "mean"))
            .sort_values(["review_band_clean", "ward_count"], ascending=[True, False])
        )
        save_csv(band_cluster, V2_REVIEW_DIR, scope_filename(scope_name, "review_band_by_dominant_cluster", "v2"))

    if "latest_election_top_party_bucket" in scope_df.columns:
        band_party = (
            scope_df.groupby(["review_band_clean", "latest_election_top_party_bucket"], dropna=False, as_index=False)
            .agg(ward_count=("WD25CD", "nunique"), mean_score=("initial_watchlist_score", "mean"))
            .sort_values(["review_band_clean", "ward_count"], ascending=[True, False])
        )
        save_csv(band_party, V2_REVIEW_DIR, scope_filename(scope_name, "review_band_by_latest_top_party", "v2"))

    print(f"\nScope {scope_name}: rows={len(scope_df)}, clean Review A/B={len(clean)}, caveated Review A/B={len(caveated)}")

# Default North West scope.
default_scope = filter_scope(scored, DEFAULT_SCOPE_NAME, DEFAULT_SCOPE_COLUMN)
make_watchlists(default_scope, DEFAULT_SCOPE_NAME)

if GENERATE_ALL_AVAILABLE_OUTPUTS:
    all_available = filter_scope(scored, "all_available", "scope_all_available")
    make_watchlists(all_available, "all_available")

if GENERATE_REGION_OUTPUTS and "analysis_region" in scored.columns:
    for region in sorted(scored["analysis_region"].dropna().unique()):
        region_scope = scored[scored["analysis_region"].eq(region)].copy()
        make_watchlists(region_scope, region)


Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_model_v2\watchlists\top_100_initial_watchlist_north_west_v2.csv
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_model_v2\watchlists\top_50_demographic_relevance_north_west_v2.csv
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_model_v2\watchlists\top_50_electoral_opportunity_north_west_v2.csv
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_model_v2\watchlists\top_50_political_openness_north_west_v2.csv
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_model_v2\watchlists\clean_watchlist_review_ab_north_west_v2.csv
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_model_v2\watchlists\caveated_watchlist_review_ab_north_west_v2.csv
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_model_v2\watchlists\demographic_build_watchlist_north_west_v2.csv
Saved: c:\Users\keena\Documents\Electoral_Tr

## 18b.7 Quick review tables

Use these displays to sanity-check whether the revised score is behaving plausibly before moving into comparison/review notebooks.


In [ ]:
print("Top 20 North West V2 watchlist")
cols = [
    "LAD25NM", "WD25NM", "analysis_region", "dominant_cluster_name",
    "latest_election_top_party_bucket", "initial_watchlist_score", "review_band_clean",
    "demographic_relevance_score", "electoral_opportunity_score", "political_openness_score", "breakthrough_complacency_score",
    "data_confidence_score", "data_confidence_note"
]
cols = [c for c in cols if c in default_scope.columns]
display(default_scope.sort_values("initial_watchlist_score", ascending=False)[cols].head(20))

print("North West council summary")
nw_council_summary = make_council_summary(default_scope)
display(nw_council_summary.head(20))


Top 20 North West V2 watchlist


,LAD25NM,WD25NM,analysis_region,dominant_cluster_name,latest_election_top_party_bucket,initial_watchlist_score,review_band_clean,demographic_relevance_score,electoral_opportunity_score,political_openness_score,breakthrough_complacency_score,data_confidence_score,data_confidence_note
3389,Burnley,Brunshaw,North West,Post-Industrial Estates / Deprived Working Com...,independent,79.763938,Review A Caveated,88.226373,75.345019,73.124807,0.000000,80,County election caveat.
3458,Lancaster,Heysham North,North West,Settled Working Families / Skilled Trades Suburbs,lab,78.611324,Review A Clean,81.165382,83.218786,62.951216,54.253246,95,No major caveat.
180,Blackpool,Waterloo,North West,Post-Industrial Estates / Deprived Working Com...,lab,75.953840,Review A Clean,82.480342,81.775542,52.212231,50.343514,95,No major caveat.
5318,Oldham,Failsworth East,North West,Settled Working Families / Skilled Trades Suburbs,lab,75.034628,Review A Clean,70.661396,81.132744,63.853265,49.322090,100,No major caveat.
5248,Bolton,Farnworth South,North West,Post-Industrial Estates / Deprived Working Com...,other,74.769921,Review A Clean,87.221810,66.303463,57.404996,0.000000,100,No major caveat.
162,Blackpool,Bloomfield,North West,Post-Industrial Estates / Deprived Working Com...,lab,73.799538,Review A Clean,95.363090,69.017176,40.869214,68.561254,95,No major caveat.
135,Warrington,Orford,North West,Post-Industrial Estates / Deprived Working Com...,lab,73.616050,Review A Clean,77.618709,76.091622,54.488060,39.107102,100,No major caveat.
1762,Cumberland,Moss Bay and Moorclose,North West,Post-Industrial Estates / Deprived Working Com...,independent,73.273791,Review A Clean,90.153490,73.886166,42.216878,0.000000,90,No major caveat.
3400,Burnley,Trinity,North West,Post-Industrial Estates / Deprived Working Com...,reform_ukip_brexit,72.851244,Review A Caveated,90.492668,60.562736,60.039959,0.000000,80,County election caveat.
3474,Lancaster,West End,North West,Settled Working Families / Skilled Trades Suburbs,reform_ukip_brexit,72.560025,Review A Caveated,83.402382,69.769666,57.753167,0.000000,80,County election caveat.


North West council summary


,LAD25CD,LAD25NM,analysis_region,ward_count,mean_score,max_score,mean_demographic,mean_electoral,mean_openness,mean_data_confidence,review_a_count,review_b_count,clean_review_ab_count,caveated_review_ab_count,high_demo_count,high_breakthrough_count,major_caveat_count,council_triage
31,E08000012,Liverpool,North West,64,50.576947,72.235706,52.813460,42.354622,39.543399,95.000000,1,5,6,0,26,15,0,Priority exploration council
23,E08000004,Oldham,North West,20,57.572562,75.034628,54.517601,53.859367,49.334368,100.000000,3,2,5,0,6,0,0,Priority exploration council
16,E07000125,Rossendale,North West,10,63.312220,70.247844,60.859318,69.682980,46.826257,94.000000,0,4,4,0,2,0,3,Priority exploration council
5,E06000050,Cheshire West and Chester,North West,45,51.476230,71.891740,54.549985,50.647215,30.758283,95.000000,1,3,4,0,13,6,0,Priority exploration council
3,E06000009,Blackpool,North West,21,59.994862,75.953840,76.622076,58.965534,23.949898,95.000000,2,1,3,0,15,7,0,Priority exploration council
20,E08000001,Bolton,North West,20,58.174833,74.769921,53.666347,59.284254,46.425343,100.000000,1,2,3,0,4,0,0,Priority exploration council
12,E07000121,Lancaster,North West,27,57.441771,78.611324,48.768641,59.490371,56.102541,85.000000,4,2,3,3,6,0,18,Priority exploration council
30,E08000011,Knowsley,North West,15,57.109560,69.552800,77.332241,38.752516,33.670085,100.000000,0,3,3,0,12,6,0,Priority exploration council
24,E08000005,Rochdale,North West,20,54.546365,70.801412,58.912574,46.848773,39.489327,100.000000,0,3,3,0,6,0,0,Priority exploration council
4,E06000049,Cheshire East,North West,52,50.966422,68.489939,48.112340,52.718639,35.246046,95.000000,0,3,3,0,5,1,0,Priority exploration council


: 